# NBA Player Similarity Recommender

Explore the bundled 2024–25 snapshot. Recommendations use 14 standardized statistics and cosine similarity. PCA and radar charts are visual aids; they do not change the ranking.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src import recommender as model
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import plotly.express as px
import plotly.graph_objects as go
model.initialize()
df, similarity = model.build_similarity_model(model.load_data())
print(f"{len(df)} eligible players, {len(model.FEATURES)} features")

## Find comparable statistical profiles

In [ ]:
selected_player = "Jayson Tatum"
recommendations = model.recommend(selected_player, top_n=5)
recommendations

## Explain the closest features

Contributions are normalized cosine terms. Displaying the six closest features does not show all contributions.

In [ ]:
compare_player = recommendations["Player"].iloc[0]
model.explain_similarity(selected_player, compare_player)

## Visualize the feature space

PCA compresses the standardized feature matrix to two dimensions. Nearby dots are an approximation, not the recommendation algorithm.

In [ ]:
X = StandardScaler().fit_transform(df[model.FEATURES])
pca = PCA(n_components=2)
coordinates = pca.fit_transform(X)
plot_df = df.assign(PC1=coordinates[:, 0], PC2=coordinates[:, 1])
print("Variance represented:", pca.explained_variance_ratio_.sum())
px.scatter(plot_df, x="PC1", y="PC2", color="Pos", hover_name="Player", template="plotly_white", title="2024–25 player profiles")

## Compare profiles

Radar values are min–max normalized only for display. Higher values are not universally better.

In [ ]:
radar = model.radar_data(selected_player, compare_player)
fig = go.Figure()
for player in [selected_player, compare_player]:
    fig.add_trace(go.Scatterpolar(r=radar[player].tolist()+radar[player].tolist()[:1], theta=radar.Feature.tolist()+radar.Feature.tolist()[:1], fill="toself", name=player))
fig.update_layout(polar=dict(radialaxis=dict(range=[0,1])), title="Stat profile comparison")
fig